# Klasy używane w algorytmach

In [32]:
class Point:
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return f"Point({self.x}, {self.y})"

class Segment:
    
    def __init__(self, point1, point2):
        self.first_point = point1
        self.second_point = point2
        
    def reversed(self):
        return Segment(self.second_point, self.first_point)

# Rysowanie punktów

In [33]:
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
import json

In [34]:
class PolygonDrawer:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ax.set_title("Kliknij, aby rysować punkty")
        self.points = []
        
        self.save_button_ax = self.fig.add_axes([0.7, 0, 0.1, 0.075])
        self.load_button_ax = self.fig.add_axes([0.81, 0, 0.1, 0.075])
        self.clear_button_ax = self.fig.add_axes([0.59, 0, 0.1, 0.075])
        self.save_button = Button(self.save_button_ax, 'Zapisz')
        self.load_button = Button(self.load_button_ax, 'Wczytaj')
        self.clear_button = Button(self.clear_button_ax, 'Wyczyść')
        self.save_button.on_clicked(self.save_polygon)
        self.load_button.on_clicked(self.load_polygon)
        self.clear_button.on_clicked(self.clear_polygon)
        
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self.onclick)
        self.draw_polygon()

    def onclick(self, event):
        if event.inaxes != self.ax:
            return
        self.points.append((event.xdata, event.ydata))
        self.draw_polygon()

    def draw_polygon(self):
        xlim, ylim = self.ax.get_xlim(), self.ax.get_ylim()
        
        self.ax.clear()
        self.ax.set_title("Kliknij, aby rysować punkty")
        
        if self.points:
            self.ax.scatter(*zip(*self.points), color='blue')
        
        self.ax.set_xlim(xlim)
        self.ax.set_ylim(ylim)
        self.fig.canvas.draw()
    
    def clear_polygon(self, event):
        self.points = []
        self.ax.clear()
        self.ax.set_title("Kliknij, aby rysować punkty")
        self.fig.canvas.draw()

    def save_polygon(self, event):
        if not self.points:
            print("Brak punktów do zapisania.")
            return
        with open('points.json', 'w') as f:
            json.dump(self.points, f)

    def load_polygon(self, event):
        try:
            with open('points.json', 'r') as f:
                self.points = json.load(f)
            self.draw_polygon()
        except FileNotFoundError:
            print("Plik 'points.json' nie istnieje.")
        except json.JSONDecodeError:
            print("Błąd wczytywania pliku - nieprawidłowy format.")

# Wizualizacja przebigu algorytmów - zapisywanie klatki

In [35]:
import os

In [36]:
def save_frame(points, hull_points, current_point, folder):
    
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    existing_files = [f for f in os.listdir(folder) if f.endswith('.png')]
    frame_idx = len(existing_files) + 1
    frame_path = os.path.join(folder, f"frame_{frame_idx:04d}.png")
    
    fig, axs = plt.subplots()
    axs.scatter([p.x for p in points], [p.y for p in points], color='black')
    
    if hull_points:
        axs.plot([p.x for p in hull_points], [p.y for p in hull_points], color='red', lw=2)
        axs.scatter([p.x for p in hull_points], [p.y for p in hull_points], color='red', s=100)
        axs.plot([hull_points[-1].x, current_point.x], 
                 [hull_points[-1].y, current_point.y], color='blue', lw=1.5)

    axs.scatter(current_point.x, current_point.y, color='blue', s=100)

    axs.set_title("Wizualizacja otoczki wypukłej")
    axs.set_xlim(min(p.x for p in points) - 1, max(p.x for p in points) + 1)
    axs.set_ylim(min(p.y for p in points) - 1, max(p.y for p in points) + 1)

    plt.savefig(frame_path)
    plt.close(fig)

# Wizualizacja algorytmu - przedstawienie wykresu

In [37]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.pyplot as plt
from PIL import Image

def load_frame_paths(folder_path):
    return sorted(
        [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')]
    )

def update(frame_path):
    axs.clear()
    img = Image.open(frame_path)
    axs.imshow(img)
    axs.axis('off')

def animate_from_folder(folder_path):
    frame_paths = load_frame_paths(folder_path)
    
    global axs
    fig, axs = plt.subplots()
    
    anim = FuncAnimation(fig, update, frames=frame_paths, repeat=False, interval=300)
    animation_html = HTML(anim.to_jshtml())
    
    clear_frames_folder(folder_path)
    
    return animation_html

def clear_frames_folder(folder_path):

    for f in os.listdir(folder_path):
        file_path = os.path.join(folder_path, f)
        if os.path.isfile(file_path):
            os.remove(file_path)